# 02 Data Combination
- Combining the datasets in different dataframes for quick analysis.

In [4]:
# Import necessary packages
import os
import warnings
from multiprocessing import Pool, Array

# Numerical and scientific computing
import numpy as np
import scipy.io as spio

# Data handling and analysis
import pandas as pd
import xarray as xr
import rioxarray
import rasterio
from rasterstats import zonal_stats

# Geospatial data handling
import geopandas as gpd

# Visualization
import seaborn as sns

# Configure settings
sns.set()  # Set Seaborn as the default style
warnings.filterwarnings('ignore')  # Suppress warnings

# Note: Without a lock provided, rioxarray's `to_raster` does not use dask to write to disk.

In [5]:
path_data = "/mnt/mountpoint/wkdir/hesse-uhi/data/"

# path for region of interest (ATKIS)
path_roi = path_data + "/roi/gemeinde_la.shp"

In [6]:
PET_class = [ 8,18,23,35] # very cold, cold, comfortable, warm, hot
HUMIDEX_class = [10,20,30,40,45] # : very low, low, medium, high, very high, extreme
DT_class =  [-3,-0.5,0.5,3] # : very low, low, medium, high, very high 

## Load Data for SV Comparison

In [7]:
path_PET_13 = path_data + "temperature/klimanalyse/PET_1000.tif" # aquisition time PET 14:00 corresponds to 13:00 LST data
path_LCZ = path_data + "other/lcz_hesse_clip.tif"
path_NDVI = path_data + "other/NDVI_crop.tif"

path_tropical_nights = path_data + "temperature/hostrada/tas_1hr_HOSTRADA_2011_2020_months-678_tropical_nights_25832_clip.tif"

path_hotspots = path_data + "temperature/hostrada/hotspots_clipped.tif"
path_coldspots = path_data + "temperature/hostrada/coldspots_clipped.tif"
path_neutralspots = path_data + "temperature/hostrada/neutralspots_clipped.tif"

path_humidex_13 = path_data + "temperature/hostrada/HUMIDEX_HOSTRADA_2011_2020_678_h13_mean_25832_clip.tif"
path_humidex_01 = path_data + "temperature/hostrada/HUMIDEX_HOSTRADA_2011_2020_678_h01_mean_25832_clip.tif"
path_humidex_13_DT = path_data + "temperature/hostrada/thermal_comfort/HUMIDEX/DT/HUMIDEX_HOSTRADA_2011_2020_678_h13_mean_buffer_25832_clip_crop.tif"
path_humidex_01_DT = path_data + "temperature/hostrada/thermal_comfort/HUMIDEX/DT/HUMIDEX_HOSTRADA_2011_2020_678_h01_mean_buffer_25832_clip_crop.tif"

path_heatindex_13 = path_data + "temperature/hostrada/thermal_comfort/Heat_Index/HEAT_INDEX_SI_HOSTRADA_2011_2020_678_h13_mean_buffer_25832_clip.tif"
path_heatindex_01 = path_data + "temperature/hostrada/thermal_comfort/Heat_Index/HEAT_INDEX_SI_HOSTRADA_2011_2020_678_h01_mean_buffer_25832_clip.tif"
path_heatindex_13_DT = path_data + "temperature/hostrada/thermal_comfort/Heat_Index/DT/crop/Heat_IndexDT_HEAT_INDEX_SI_HOSTRADA_2011_2020_678_h13_mean_buffer_25832_clip_radius_100_2024-07-17_12-52-36_clip_crop.tif"
path_heatindex_01_DT = path_data + "temperature/hostrada/thermal_comfort/Heat_Index/DT/crop/Heat_IndexDT_HEAT_INDEX_SI_HOSTRADA_2011_2020_678_h01_mean_buffer_25832_clip_radius_100_2024-07-17_12-59-03_clip_crop.tif"

path_discomfortindex_13 = path_data + "temperature/hostrada/HUMIDEX_HOSTRADA_2011_2020_678_h13_mean_25832_clip.tif"
path_discomfortindex_01 = path_data + "temperature/hostrada/HUMIDEX_HOSTRADA_2011_2020_678_h01_mean_25832_clip.tif"
path_discomfortindex_13_DT = path_data + "temperature/hostrada/DT/DT_HUMIDEX_HOSTRADA_2011_2020_678_h13_mean_buffer_25832_clip_radius_100_2024-06-11_10-40-31_clip.tif"
path_discomfortindex_01_DT = path_data + "temperature/hostrada/DT/DT_HUMIDEX_HOSTRADA_2011_2020_678_h01_mean_buffer_25832_clip_radius_100_2024-06-11_11-19-55_clip.tif"

path_90th = path_data + "temperature/hostrada/heatwave/tas_1hr_HOSTRADA_2011_2020_months-678_90th_25832_clip.tif"
path_95th = path_data + "temperature/hostrada/heatwave/tas_1hr_HOSTRADA_2011_2020_months-678_95th_25832_clip.tif"
path_heatwave_days_95th_28 = path_data + "temperature/hostrada/heatwave/tas_1hr_HOSTRADA_2011_2020_months-678_heat_wave_days_95th_28_25832_clip.tif"
path_hot_day = path_data + "temperature/hostrada/heatwave/tas_1hr_HOSTRADA_2011_2020_months-678_hot_day_25832_clip.tif"
path_very_hot_day = path_data + "temperature/hostrada/heatwave/tas_1hr_HOSTRADA_2011_2020_months-678_very_hot_day_25832_clip.tif"

path_people = path_data + "geoportal/bevoelkerung_2011.tif"
path_people_u18 = path_data + "geoportal/anteil_unter_18.tif"
path_people_ab65 = path_data + "geoportal/anteil_ab_65.tif"

In [8]:
PET_13 =   rasterio.open(path_PET_13).read(1)
PET_13[PET_13==9999]=np.nan
PET_13 = PET_13.flatten()

NDVI = rasterio.open(path_NDVI).read(1).flatten()

LCZ = rasterio.open(path_LCZ).read(1).flatten()

tropical_nights = rasterio.open(path_tropical_nights).read(1).flatten()
hotspots = rasterio.open(path_hotspots).read(1).flatten()
coldspots = rasterio.open(path_coldspots).read(1).flatten()
neutralspots = rasterio.open(path_neutralspots).read(1).flatten()

Ta_90th = rasterio.open(path_90th).read(1).flatten()
Ta_95th = rasterio.open(path_95th).read(1).flatten()
HeatWave_days = rasterio.open(path_heatwave_days_95th_28).read(1).flatten()
hot_day= rasterio.open(path_hot_day).read(1).flatten()
very_hot_day= rasterio.open(path_very_hot_day).read(1).flatten()

humidex_13 = rasterio.open(path_humidex_13).read(1).flatten()
humidex_01 = rasterio.open(path_humidex_01).read(1).flatten()
humidex_13_DT = rasterio.open(path_humidex_13_DT).read(1).flatten()
humidex_01_DT = rasterio.open(path_humidex_01_DT).read(1).flatten()

heatindex_13 = rasterio.open(path_heatindex_13).read(1).flatten()
heatindex_01 = rasterio.open(path_heatindex_01).read(1).flatten()
heatindex_13_DT = rasterio.open(path_heatindex_13_DT).read(1).flatten()
heatindex_01_DT = rasterio.open(path_heatindex_01_DT).read(1).flatten()

people =   rasterio.open(path_people).read(1).flatten()
people_u18 =   rasterio.open(path_people_u18).read(1).flatten()
people_ab65 =   rasterio.open(path_people_ab65).read(1).flatten()

In [9]:
path_LST_01 = path_data + "temperature/modis/DT/DT_MODIS_aqua_night_2011-1-1_2021-1-1_radius_100_radius_2024-03-07_14-10-21.tif"
path_LST_10 = path_data + "temperature/modis/DT/DT_MODIS_terra_day_2011-1-1_2021-1-1_radius_100_radius_2024-03-07_15-30-33.tif"
path_LST_13 = path_data + "temperature/modis/DT/DT_MODIS_aqua_day_2011-1-1_2021-1-1_radius_100_radius_2024-03-07_15-00-17.tif"
path_LST_22 = path_data + "temperature/modis/DT/DT_MODIS_terra_night_2011-1-1_2021-1-1_radius_100_radius_2024-03-07_13-11-03.tif"

path_Ta_01_mean = path_data + "temperature/hostrada/hourly/DT/DT_tas_1hr_HOSTRADA_h1_mean_25832_clip_radius_100_2024-05-14_09-57-52_clip_crop.tif"
path_Ta_10_mean = path_data + "temperature/hostrada/hourly/DT/DT_tas_1hr_HOSTRADA_h10_mean_25832_clip_radius_100_2024-05-13_14-49-17_clip_crop.tif"
path_Ta_13_mean = path_data + "temperature/hostrada/hourly/DT/DT_tas_1hr_HOSTRADA_h13_mean_25832_clip_radius_100_2024-05-07_13-52-55_clip_crop.tif"
path_Ta_22_mean = path_data + "temperature/hostrada/hourly/DT/DT_tas_1hr_HOSTRADA_h22_mean_25832_clip_radius_100_2024-05-14_10-26-28_clip_crop.tif"

#path_Ta_01_90th = path_data + "temperature/hostrada/decade/DT/DT_tas_1hr_HOSTRADA_2011_2020_678_h1_90th_25832_clip_radius_100_radius_2024-05-07_09-38-19_clip.tif"
#path_Ta_10_90th = path_data + "temperature/hostrada/decade/DT/DT_tas_1hr_HOSTRADA_2011_2020_678_h10_90th_25832_clip_radius_100_radius_2024-05-07_09-38-19_clip.tif"
#path_Ta_13_90th = path_data + "temperature/hostrada/decade/DT/DT_tas_1hr_HOSTRADA_2011_2020_678_h13_90th_25832_clip_radius_100_radius_2024-05-07_09-38-19_clip.tif"
#path_Ta_20_90th = path_data + "temperature/hostrada/decade/DT/DT_tas_1hr_HOSTRADA_2011_2020_678_h20_90th_25832_clip_radius_100_radius_2024-05-07_09-38-19_clip.tif"

In [10]:
# Open the geotiffs
LST_01 = rasterio.open(path_LST_01).read(1).flatten()
LST_10 = rasterio.open(path_LST_10).read(1).flatten()
LST_13 = rasterio.open(path_LST_13).read(1).flatten()
LST_22 = rasterio.open(path_LST_22).read(1).flatten()

Ta_01 = rasterio.open(path_Ta_01_mean).read(1).flatten()
Ta_10 = rasterio.open(path_Ta_10_mean).read(1).flatten()
Ta_13 = rasterio.open(path_Ta_13_mean).read(1).flatten()
Ta_22 = rasterio.open(path_Ta_22_mean).read(1).flatten()

## Create Dataframes

In [ ]:
# Create a Pandas DataFrame
df = pd.DataFrame({
    'LST_01': LST_01,'LST_10': LST_10,'LST_13': LST_13,'LST_22': LST_22, 
    'Ta_01': Ta_01, 'Ta_10': Ta_10, 'Ta_13': Ta_13, 'Ta_22': Ta_22, 
    'PET_13': PET_13, 'HUMIDEX_13':humidex_13, 'HUMIDEX_01':humidex_01, 'HUMIDEX_13_DT':humidex_13_DT, 'HUMIDEX_01_DT':humidex_01_DT,
    #Heat_Index_13': heatindex_13, 'Heat_Index_01': heatindex_01, 'Heat_Index_13_DT': heatindex_13_DT, 'Heat_Index_01': heatindex_01_DT,
    'LCZ': LCZ,
    'NDVI': NDVI,
    'tropical_nights':tropical_nights,'hotspots':hotspots,'coldspots':coldspots,'neutralspots':neutralspots,
    "Ta_90th":Ta_90th, "Ta_95th":Ta_95th, "heatwave":HeatWave_days, "hot_day":hot_day,"very_hot_day":very_hot_day,
    'people': people,'people_u18': people_u18,'people_ab65': people_ab65
})#.dropna() change if needed
df.head()

In [24]:
df.describe()

,LST_01,LST_10,LST_13,LST_22,Ta_01,Ta_10,Ta_13,Ta_22,PET_13,HUMIDEX_13,...,coldspots,neutralspots,Ta_90th,Ta_95th,heatwave,hot_day,very_hot_day,people,people_u18,people_ab65
count,20668.000000,20664.000000,20667.000000,20667.000000,20662.000000,20663.000000,20664.000000,20659.000000,21762.000000,20987.000000,...,21089.000000,21089.000000,21089.000000,21089.000000,21089.000000,21089.000000,21089.000000,43674.000000,43674.000000,43674.000000
mean,-0.106679,-1.491197,-1.828938,-0.147084,-0.275848,-0.157657,-0.104510,-0.283548,30.896830,24.035001,...,18.905788,76.602929,24.714522,26.964348,87.279482,73.666698,5.920480,247.795004,1.032834,1.226542
std,0.836245,1.528210,1.718430,0.898849,0.399015,0.311827,0.356003,0.389270,8.541185,1.323658,...,26.531632,29.087071,1.070627,1.145362,32.446209,31.599391,5.898402,754.583180,2.111358,2.341021
min,-3.003061,-7.597969,-7.828353,-3.137232,-1.635532,-2.280997,-2.582196,-1.828748,13.171896,18.005505,...,0.000000,0.000000,20.500000,22.500000,8.000000,4.000000,0.000000,-1.000000,-9.000000,-9.000000
25%,-0.679182,-2.544306,-3.004951,-0.705104,-0.514126,-0.323135,-0.290466,-0.533058,22.663214,23.215513,...,0.000000,54.166667,24.000000,26.200000,63.000000,53.000000,2.000000,-1.000000,-1.000000,-1.000000
50%,-0.127242,-1.457393,-1.808308,-0.126600,-0.295296,-0.137238,-0.080360,-0.317241,29.735249,23.934646,...,0.000000,91.666667,24.500000,26.800000,83.000000,66.000000,4.000000,4.000000,1.000000,1.000000
75%,0.453211,-0.462378,-0.701530,0.413841,-0.061455,0.030622,0.110682,-0.077099,39.420002,24.869337,...,37.500000,100.000000,25.200000,27.500000,114.000000,87.000000,7.000000,141.000000,3.000000,3.000000
max,3.413033,5.821229,6.172398,4.910873,2.857384,1.394490,1.559876,2.629910,50.534672,27.146401,...,100.000000,100.000000,27.900000,30.200000,153.000000,175.000000,36.000000,19198.000000,5.000000,5.000000


In [ ]:
PET_13_classes = np.digitize(PET_13.flatten(), PET_class, right=True)

LST_01_classes = np.digitize(LST_01.flatten(), DT_class, right=True)
LST_10_classes = np.digitize(LST_10.flatten(), DT_class, right=True)
LST_13_classes = np.digitize(LST_13.flatten(), DT_class, right=True)
LST_22_classes = np.digitize(LST_22.flatten(), DT_class, right=True)

Ta_01_classes = np.digitize(Ta_01.flatten(), DT_class, right=True)
Ta_10_classes = np.digitize(Ta_10.flatten(), DT_class, right=True)
Ta_13_classes = np.digitize(Ta_13.flatten(), DT_class, right=True)
Ta_22_classes = np.digitize(Ta_22.flatten(), DT_class, right=True)


# Create a Pandas DataFrame
df_classes = pd.DataFrame({
    'LST_01': LST_01_classes,'LST_10': LST_10_classes,'LST_13': LST_13_classes,'LST_22': LST_22_classes, 
    'Ta_01': Ta_01_classes, 'Ta_10': Ta_10_classes, 'Ta_13': Ta_13_classes, 'Ta_22': Ta_22_classes, 
    'PET_13': PET_13_classes, 'HUMIDEX_13':humidex_13, 'HUMIDEX_01':humidex_01, 'HUMIDEX_13_DT':humidex_13_DT, 'HUMIDEX_01_DT':humidex_01_DT,
    'LCZ': LCZ,
    'NDVI': NDVI,
    'tropical_nights':tropical_nights,'hotspots':hotspots,'coldspots':coldspots,'neutralspots':neutralspots,
    'people': people,'people_u18': people_u18,'people_ab65': people_ab65,
})#.dropna() not correctly working

# Delete same rows in the second DataFrame
df_classes = df_classes.loc[df.index]

df_classes.describe()

In [ ]:
df_test = df_classes.copy()

tag = "_class"

# Add tag to all column names
df_test.columns = [col + tag for col in df_test.columns]
df_combined = pd.concat([df, df_test], axis=1)


df_combined.describe()

In [27]:
df_combined.to_csv('df_combined_parameters_nan.csv',sep=";")

## 24 Hour Analysis

In [ ]:
# options: HUMIDEX DT, HEAT INDEX DT,  HOSTRADA DT, HOSTRADA, HEAT INDEX
parameter="HUMIDEX_DT"

if parameter == "HUMIDEX_DT":
    # load data
    files_hourly = []
    for i in range(24):
        for file in os.listdir(path_data+'temperature/hostrada/thermal_comfort/HUMIDEX/DT/'):
            string = 'HUMIDEX_HOSTRADA_2011_2020_678_h'+f'{i:02d}'+'_mean_buffer'
            if string in file:
                print(file)
                files_hourly.append(os.path.join(path_data+'temperature/hostrada/thermal_comfort/HUMIDEX/DT', file))
elif parameter == "Heat_Index_DT":
    # load data
    files_hourly = []
    for i in range(24):
        for file in os.listdir(path_data+'temperature/hostrada/thermal_comfort/Heat_Index/DT/'):
            string = 'Heat_IndexDT_HEAT_INDEX_SI_HOSTRADA_2011_2020_678_h'+f'{i:02d}'+'_mean_buffer'
            if string in file:
                print(file)
                files_hourly.append(os.path.join(path_data+'temperature/hostrada/thermal_comfort/Heat_Index/DT', file))
elif parameter == "HOSTRADA_DT":
    # load data
    files_hourly = []
    for i in range(24):
        for file in os.listdir(path_data+'temperature/hostrada/hourly/DT/mean/'):
            string = 'DT_tas_1hr_HOSTRADA_h'+f'{i:d}'+'_mean_25832_clip_radius_100'
            if string in file:
                print(file)
                files_hourly.append(os.path.join(path_data+'temperature/hostrada/hourly/DT/mean', file))
elif parameter == "HOSTRADA_DT_90th":
    # load data
    files_hourly = []
    for i in range(24):
        for file in os.listdir(path_data+'temperature/hostrada/hourly/DT/90th/'):
            string = 'DT_tas_1hr_HOSTRADA_h'+f'{i:d}'+'_90th_25832_clip_radius_100'
            if string in file:
                print(file)
                files_hourly.append(os.path.join(path_data+'temperature/hostrada/hourly/DT/90th', file))
elif parameter == "HOSTRADA":
    # load data
    files_hourly = []
    for i in range(24):
        for file in os.listdir(path_data+'temperature/hostrada/hourly/'):
            string = 'tas_1hr_HOSTRADA_h'+f'{i:d}'+'_mean_25832_clip.tif'
            if string in file:
                print(file)
                files_hourly.append(os.path.join(path_data+'temperature/hostrada/hourly/', file))

elif parameter == "Heat_Index":
    # load data
    files_hourly = []
    for i in range(24):
        for file in os.listdir(path_data+'temperature/hostrada/thermal_comfort/Heat_Index/'):
            string = 'HEAT_INDEX_SI_HOSTRADA_2011_2020_678_h'+f'{i:02d}'+'_mean_buffer_25832_clip'
            if string in file:
                print(file)
                files_hourly.append(os.path.join(path_data+'temperature/hostrada/thermal_comfort/Heat_Index/', file))


In [83]:
data_hourly = xr.open_mfdataset(
    files_hourly,
    parallel=True,
    concat_dim="band",
    combine="nested",
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

data_hourly

<xarray.Dataset>
Dimensions:      (band: 24, x: 173, y: 251)
Coordinates:
  * band         (band) int64 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
  * x            (x) float64 4.135e+05 4.145e+05 ... 5.845e+05 5.855e+05
  * y            (y) float64 5.722e+06 5.722e+06 ... 5.474e+06 5.472e+06
    spatial_ref  int64 ...
Data variables:
    band_data    (band, y, x) float64 dask.array<chunksize=(1, 5, 173), meta=np.ndarray>

In [84]:
data_24h = []
col_24h =[]
for h in range(0,24):
    data_h = np.array(data_hourly["band_data"][h,:,:]).flatten()
    data_24h.append(data_h)
    col_24h.append(f'{h:02}'+":00")
    #col_24h.append(h)

df_24h =  pd.DataFrame(np.column_stack(data_24h), columns=col_24h).dropna()
df_24h.head()

,00:00,01:00,02:00,03:00,04:00,05:00,06:00,07:00,08:00,09:00,...,14:00,15:00,16:00,17:00,18:00,19:00,20:00,21:00,22:00,23:00
119,0.1,0.0,0.05,0.1,0.1,0.0,0.0,0.0,-0.1,-0.1,...,-0.1,-0.1,-0.1,0.0,0.0,0.1,0.2,0.2,0.2,0.1
120,0.0,0.0,-0.20,0.0,0.1,0.0,0.0,0.0,-0.1,-0.2,...,-0.2,-0.1,-0.2,-0.1,-0.1,0.0,-0.1,-0.2,-0.3,-0.1
291,-0.1,-0.2,-0.20,-0.2,-0.2,-0.2,-0.2,-0.2,-0.2,-0.1,...,-0.1,-0.1,0.0,0.0,-0.1,0.1,0.0,0.0,-0.1,-0.1
292,-0.6,-0.7,-0.70,-0.7,-0.6,-0.5,-0.5,-0.3,-0.2,-0.2,...,-0.1,-0.1,-0.1,-0.1,-0.1,-0.2,-0.2,-0.4,-0.4,-0.5
293,-0.4,-0.4,-0.50,-0.4,-0.4,-0.3,-0.3,-0.2,-0.3,-0.2,...,-0.2,-0.2,-0.2,-0.1,-0.1,-0.2,-0.4,-0.5,-0.6,-0.5


In [ ]:
df_combined_24h = pd.concat([df_24h, df], axis=1)
df_combined_24h

In [86]:
df["index"]=df.index
df.head()

df_long = df_24h.reset_index().melt(id_vars='index', var_name='Column', value_name='Temperature')
df_long 

,index,Column,Temperature
0,119,00:00,0.1
1,120,00:00,0.0
2,291,00:00,-0.1
3,292,00:00,-0.6
4,293,00:00,-0.4
...,...,...,...
493795,43153,23:00,0.0
493796,43154,23:00,0.6
493797,43323,23:00,-0.6
493798,43325,23:00,0.7


In [ ]:
df_long_extra = df_long.merge(df, on="index")

# Define thresholds for city sizes
bins = [-float('inf'), 250, 500, 1000, 2000,5000,10000, float('inf')]
labels = ['0', '1', '2', '3', '4', '5', '6']

# Create the city_size column based on the thresholds
df_long_extra['city_size'] = pd.cut(df_long_extra['people'], bins=bins, labels=labels)

u18_percentage_map = {
    -9: 0.0,  # No data, assuming 0%
    -1: 0.0,  # No data, assuming 0%
    1: 0.0,   # 0%
    2: 0.075, # 0%-15% average to 7.5%
    3: 0.175, # 15%-20% average to 17.5%
    4: 0.225, # 20%-25% average to 22.5%
    5: 0.25   # >=25%, assuming minimum 25%
}

# Apply the mapping to calculate the percentage of people under 18
df_long_extra['u18_percentage'] = df_long_extra['people_u18'].map(u18_percentage_map)

# Calculate the number of people under 18
df_long_extra['people_u18_count'] = df_long_extra['people'] * df_long_extra['u18_percentage']

# Create the city_size column based on the thresholds
df_long_extra['people_u18_count'] = pd.cut(df_long_extra['people'], bins=bins, labels=labels)

df_long_extra

In [88]:
df_long_extra.to_csv('df_combined_output_24h_'+parameter+'.csv',sep=";", index=False)

### Plotting 24 hours

### Calculation Hot- & Cold Spots

In [ ]:
# set categories for the data: very low, low, medium, high, very high

# set the bounds for each category [DT_class, ]
bounds = xr.DataArray(DT_class, dims=['k'])
# check for how much of the hour the data is in each category

# create a new data array with the categories
data_categories = []
for i in range(24):
    data_test_2 = xr.apply_ufunc(
        np.digitize, np.array(data_hourly["band_data"][i,:,:]), bounds, dask="parallelized",input_core_dims=[["y", "x"], ['k']], output_core_dims=[['y', 'x']], 
    )

    data_categories.append(xr.DataArray(name="uhi", data=data_test_2.where(data_test_2 != 4 , np.nan), dims=["y", "x"])) #ansonsten ungleich 5

for i, d in enumerate(data_categories):
    d.coords.update(data_hourly["band_data"][1,:,:].coords)

In [ ]:
combined_dataset = xr.concat(data_categories,dim='hour')#.transpose('hour','x', 'y')

In [ ]:
np.unique(combined_dataset.values)

In [ ]:
neutral = (((combined_dataset == 2).sum(dim='hour')/24)*100) #neutral = (((combined_dataset == 2).sum(dim='hour')/24)*100)
hotspots = (((combined_dataset > 2).sum(dim='hour')/24)*100) # hotspots = (((combined_dataset == 3).sum(dim='hour')/24)*100) #
coldspots = (((combined_dataset < 2).sum(dim='hour')/24)*100)# coldspots = (((combined_dataset == 1).sum(dim='hour')/24)*100)

In [ ]:
neutral.plot()

In [ ]:
hotspots.plot()

In [ ]:
coldspots.plot()

In [ ]:
hotspots.rio.set_spatial_dims("x", "y")
hotspots.rio.set_crs("EPSG:25832")

coldspots.rio.set_spatial_dims("x", "y")
coldspots.rio.set_crs("EPSG:25832")

# crop to ROI
Hesse = gpd.read_file(path_roi).dissolve(by='erzeugt_am').to_crs("EPSG:25832")

hotspots.rio.to_raster(path_data+'temperature/hostrada/hotspots_HUMIDEX_DT.tif' )
raster_data = rioxarray.open_rasterio(path_data+'temperature/hostrada/hotspots_HUMIDEX_DT.tif' )
raster_data_clipped = raster_data.rio.clip(Hesse.geometry, Hesse.crs)
raster_data_clipped.rio.to_raster(path_data+'temperature/hostrada/hotspots_HUMIDEX_DT.tif' )

coldspots.rio.to_raster(path_data+'temperature/hostrada/coldspots_HUMIDEX_DT.tif' )
raster_data = rioxarray.open_rasterio(path_data+'temperature/hostrada/coldspots_HUMIDEX_DT.tif' )
raster_data_clipped = raster_data.rio.clip(Hesse.geometry, Hesse.crs)
raster_data_clipped.rio.to_raster(path_data+'temperature/hostrada/coldspots_HUMIDEX_DT.tif' )

In [ ]:
neutral.rio.set_spatial_dims("x", "y")
neutral.rio.set_crs("EPSG:25832")

neutral.rio.to_raster(path_data+'temperature/hostrada/neutralspots_HUMIDEX_DT.tif' )
raster_data = rioxarray.open_rasterio(path_data+'temperature/hostrada/neutralspots_HUMIDEX_DT.tif' )
raster_data_clipped = raster_data.rio.clip(Hesse.geometry, Hesse.crs)
raster_data_clipped.rio.to_raster(path_data+'temperature/hostrada/neutralspots_HUMIDEX_DT.tif' )

## Districts
not used

In [ ]:
district = gpd.read_file(path_roi).to_crs("EPSG:25832")

district['kreis_bz'] = district['kreis_bz'].str.replace('Ã¼', 'ü')
district['kreis_bz'] = district['kreis_bz'].str.replace('ÃŸ', 'ß')
district['kreis_bz'] = district['kreis_bz'].str.replace('Ã¤', 'ä')
district['kreis_bz'] = district['kreis_bz'].str.replace('Ã¶', 'ö')

district['regbez_bz'] = district['regbez_bz'].str.replace('Ã¼', 'ü')
district['regbez_bz'] = district['regbez_bz'].str.replace('ÃŸ', 'ß')
district['regbez_bz'] = district['regbez_bz'].str.replace('Ã¤', 'ä')
district['regbez_bz'] = district['regbez_bz'].str.replace('Ã¶', 'ö')

district['gmde_bz'] = district['gmde_bz'].str.replace('Ã¼', 'ü')
district['gmde_bz'] = district['gmde_bz'].str.replace('ÃŸ', 'ß')
district['gmde_bz'] = district['gmde_bz'].str.replace('Ã¤', 'ä')
district['gmde_bz'] = district['gmde_bz'].str.replace('Ã¶', 'ö')

path_people = path_data + "geoportal/bevoelkerung_2011.tif"

array =   rasterio.open(path_people).read(1)
affine = rasterio.open(path_people).transform
zs = zonal_stats(district, array, affine=affine, stats=['sum'])
values_mean = [feature["sum"] for feature in zs]

district["people"] = values_mean

district = district.sort_values(
        by="people", ascending=False
    ).reset_index(drop=True)

district

In [ ]:
stadt_klein = district.loc[(district['people'] >= 5000) & (district['people'] < 20000)]
stadt_mittel =district.loc[(district['people'] >= 20000) & (district['people'] < 100000)]
stadt_groß =district.loc[(district['people'] >= 100000)]